In [ ]:
from pyspark.sql import functions as F

table_keys = {
    "rental_offices": ["officename"],
    "cars": ["plate"],
    "have_optional": ["rentalid"],
    "drivers": ["licensenumber"],
    "rentals": ["plate", "pickupdate"],
    "drive": ["licensenumber", "plate", "pickupdate"],
    "insurances": ["risk", "plate", "pickupdate"],
    "payments": ["plate", "pickupdate"],
}

In [ ]:
def validate_keys(df, keys):
    invalid_key = F.lit(False)

    for key in keys:
        invalid_key = invalid_key | (
            F.col(key).isNull()
            | (F.trim(F.col(key).cast("string")) == "")
        )

    missing = df.filter(invalid_key).count()

    duplicates = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    return df.count(), missing, duplicates

In [ ]:
raw_base = "abfss://raw@stcarrentaldev.dfs.core.windows.net"

raw_tables = {}
results = []

for table, keys in table_keys.items():
    path = f"{raw_base}/{table}/{table.upper()}.parquet"
    df = spark.read.parquet(path)

    raw_tables[table] = df
    rows, missing, duplicates = validate_keys(df, keys)

    results.append((table, rows, missing, duplicates))

display(
    spark.createDataFrame(
        results,
        ["table", "rows", "missing_keys", "duplicate_key_groups"]
    )
)

if any(missing or duplicates for _, _, missing, duplicates in results):
    raise ValueError("Key validation failed. Check the results above.")





In [ ]:
# Each entry: child table, parent table, child columns, parent columns
relationships = [
    ("have_optional", "cars", ["plate"], ["plate"]),
    ("rentals", "cars", ["plate"], ["plate"]),
    ("rentals", "rental_offices", ["pickupplace"], ["officename"]),
    ("rentals", "rental_offices", ["dropoffplace"], ["officename"]),
    ("drive", "drivers", ["licensenumber"], ["licensenumber"]),
    ("drive", "rentals", ["plate", "pickupdate"], ["plate", "pickupdate"]),
    ("insurances", "rentals", ["plate", "pickupdate"], ["plate", "pickupdate"]),
    ("payments", "rentals", ["plate", "pickupdate"], ["plate", "pickupdate"]),
]

def count_missing_references(child, parent, child_keys, parent_keys):
    parent_df = (
        raw_tables[parent]
        .select(*parent_keys)
        .toDF(*child_keys)
    )

    return (
        raw_tables[child]
        .join(parent_df, on=child_keys, how="left_anti")
        .count()
    )

In [ ]:
business_checks = []

for child, parent, child_keys, parent_keys in relationships:
    failures = count_missing_references(
        child, parent, child_keys, parent_keys
    )
    label = f"{child} ({', '.join(child_keys)}) -> {parent}"
    business_checks.append((label, failures))

invalid_dates = raw_tables["rentals"].filter(
    F.col("dropoffdate").isNull()
    | (F.col("pickupdate") >= F.col("dropoffdate"))
).count()

business_checks.append(("Invalid rental dates", invalid_dates))

display(
    spark.createDataFrame(
        business_checks,
        ["check", "failed_rows"]
    )
)

if any(count > 0 for _, count in business_checks):
    raise ValueError("Business validation failed. Check the results above.")

In [ ]:
column_names = {
    "officename": "office_name",
    "registrationdate": "registration_date",
    "rentalid": "rental_id",
    "licensenumber": "license_number",
    "licenseexpiration": "license_expiration",
    "drivername": "driver_name",
    "birthdate": "birth_date",
    "pickupdate": "pickup_date",
    "dropoffdate": "dropoff_date",
    "pickupplace": "pickup_place",
    "dropoffplace": "dropoff_place",
    "paymentmode": "payment_mode",
}

In [ ]:
if any(missing or duplicates for _, _, missing, duplicates in results):
    raise ValueError("Key checks must pass before writing.")

if any(failures for _, failures in business_checks):
    raise ValueError("Business checks must pass before writing.")

curated_base = "abfss://curated@stcarrentaldev.dfs.core.windows.net"
expected_counts = {table: rows for table, rows, _, _ in results}

write_results = []

for table, df in raw_tables.items():
    clean_df = df.select(
        *[
            F.col(column).alias(column_names.get(column, column))
            for column in df.columns
        ]
    ).withColumn("processed_at", F.current_timestamp())

    destination = f"{curated_base}/{table}"

    (
        clean_df.write
        .format("delta")
        .mode("overwrite")
        .save(destination)
    )

    saved_count = spark.read.format("delta").load(destination).count()
    expected_count = expected_counts[table]

    if saved_count != expected_count:
        raise ValueError(f"Row count mismatch for {table}")

    write_results.append((table, expected_count, saved_count))

display(
    spark.createDataFrame(
        write_results,
        ["table", "raw_rows", "curated_rows"]
    )
)

In [ ]:
catalog = "dbw_carrental_dev_7405614156846449"
curated_base = "abfss://curated@stcarrentaldev.dfs.core.windows.net"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.curated")

for table in table_keys:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS `{catalog}`.curated.`{table}`
        USING DELTA
        LOCATION '{curated_base}/{table}'
    """)

display(spark.sql(f"SHOW TABLES IN `{catalog}`.curated"))